# Optimization 6 - Long history + sales-value weighting (Google Colab)

Adds a third model to the TSO-DTF ensemble. It is **Opt 4 with two changes**:
1. **2 years of training history** instead of 1 year.
2. **Sales-value weighting**: each product's rows are weighted by the square root of its revenue in the 28 days *before* the test period (no test data is used).
3. Averaged over **2 random seeds**.

Same stores, same test days, same products and same inputs as all other methods.

**How to run:** Runtime -> Change runtime type -> High-RAM if available. Then **Runtime -> Run all**. Takes about 45-60 minutes.
At the end a file `opt6_results.zip` downloads.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q lightgbm optuna cmaes pyarrow


In [ ]:
import os
import pandas as pd, numpy as np, lightgbm as lgb, optuna, glob, gc, shutil, time
import warnings; warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "/content/drive/MyDrive/m5_data"
OUT_DIR = "/content/drive/MyDrive/m5_results"
WORK_DIR = "/content/m5_work"
if not os.path.exists(f"{DATA_DIR}/sell_prices.csv"):
    raise FileNotFoundError("Put the M5 CSV files in My Drive/m5_data")
MERGED_DIR, FEATS_DIR = f"{WORK_DIR}/merged", f"{WORK_DIR}/features_730"
for d in [OUT_DIR, MERGED_DIR, FEATS_DIR]:
    os.makedirs(d, exist_ok=True)

store_list = ["CA_1", "TX_1", "WI_1"]      # same stores as the baseline
FORECAST_HORIZON = 7
TEST_DAYS = 90
TRAIN_WINDOW_DAYS = 730                   # Opt 6: two years of history (other methods: 365)
VALID_DAYS = 56
SCORE_HIST_DAYS = 365                     # RMSSE scale uses the same 365-day history as every other method
SEEDS = [42, 7]                           # seed averaging
RANDOM_SEED = 42
print("Data:", DATA_DIR, "| Results:", OUT_DIR)


## Step 1 - Data preparation

In [ ]:
cal = pd.read_csv(f"{DATA_DIR}/calendar.csv", parse_dates=["date"])
if "d" not in cal.columns:
    cal["d"] = ["d_" + str(i + 1) for i in range(len(cal))]
prices = pd.read_csv(f"{DATA_DIR}/sell_prices.csv")
sales_path = f"{DATA_DIR}/sales_train_evaluation.csv"
id_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in pd.read_csv(sales_path, nrows=0).columns if c.startswith("d_")]
snap_cols = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}

# event proximity (calendar is known in advance, so no leakage)
ev_dates = cal.loc[cal["event_name_1"].notna(), "date"].sort_values().values
cal_dates = cal["date"].values
nxt = np.searchsorted(ev_dates, cal_dates, side="left")
prv = np.searchsorted(ev_dates, cal_dates, side="right") - 1
cal["days_to_event"] = [(ev_dates[i] - d).astype("timedelta64[D]").astype(int) if i < len(ev_dates) else 99 for i, d in zip(nxt, cal_dates)]
cal["days_since_event"] = [(d - ev_dates[i]).astype("timedelta64[D]").astype(int) if i >= 0 else 99 for i, d in zip(prv, cal_dates)]
cal[["days_to_event", "days_since_event"]] = cal[["days_to_event", "days_since_event"]].clip(upper=99)

for store_id in store_list:
    out_path = f"{MERGED_DIR}/{store_id}.parquet"
    if os.path.exists(out_path):
        print(f"{store_id}: already prepared"); continue
    parts = [ch[ch["store_id"] == store_id] for ch in pd.read_csv(sales_path, chunksize=2000)]
    s = pd.concat(parts, ignore_index=True); del parts
    state = s["state_id"].iloc[0]
    long = s.melt(id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="sales")
    long["sales"] = long["sales"].astype("float32"); del s; gc.collect()
    long = long.merge(cal[["d", "date", "wm_yr_wk", "wday", "event_name_1", "days_to_event", "days_since_event",
                           snap_cols[state]]], on="d", how="left").rename(columns={snap_cols[state]: "snap"})
    long = long.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    long.to_parquet(out_path, index=False)
    print(f"{store_id}: saved {len(long):,} rows"); del long; gc.collect()

max_date = cal["date"].max()
test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)
print("Test period starts", test_start.date())


## Step 2 - Features (same builder as Optimizations 4 and 5)

In [ ]:
FEATURE_GROUPS = {
    # ---- baseline (base paper) groups ----
    "lags_1_7":        [f"sales_lag_{l}" for l in range(1, 8)],
    "lags_8_14":       [f"sales_lag_{l}" for l in range(8, 15)],
    "rolling_h7":      [f"rolling_{s}_{w}" for w in [7, 14, 30, 60, 180] for s in ["mean", "stdv"]],
    "calendar":        ["day_of_week", "day_of_month", "week_of_year", "month", "quarter", "end_of_week"],
    "price":           ["sell_price", "price_ratio_to_mean", "price_ratio_to_max"],
    "promo_event":     ["snap", "is_event"],
    # ---- new groups ----
    "weekly_lags":     ["sales_lag_21", "sales_lag_28", "sales_lag_35", "sales_lag_42"],
    "same_weekday_avg":["dow_mean_4", "dow_mean_8"],
    "intermittency":   ["zero_share_28", "days_since_sale", "nonzero_mean_28", "nonzero_count_28"],
    "short_rolling":   ["recent_mean_3", "recent_mean_7", "recent_mean_28"],
    "price_dynamics":  ["price_change_1w", "price_rel_dept"],
    "event_proximity": ["days_to_event", "days_since_event"],
    "item_encoding":   ["item_mean_hist", "item_std_hist"],
}
BASELINE_GROUPS = ["lags_1_7", "lags_8_14", "rolling_h7", "calendar", "price", "promo_event"]
NEW_GROUPS = [g for g in FEATURE_GROUPS if g not in BASELINE_GROUPS]
CAT_FEATURES = ["dept_id", "cat_id"]
KEY = ["store_id", "item_id"]

def build_features_ext(df, horizon=FORECAST_HORIZON):
    df = df.sort_values(["item_id", "date"]).reset_index(drop=True)
    g = df.groupby("item_id", sort=False)["sales"]
    for l in list(range(1, 15)) + [21, 28, 35, 42, 49, 56]:
        df[f"sales_lag_{l}"] = g.shift(l).astype("float32")
    # baseline rolling features (shifted by the horizon, as in the base paper)
    df["_sh"] = g.shift(horizon)
    gs = df.groupby("item_id", sort=False)["_sh"]
    for w in [7, 14, 30, 60, 180]:
        df[f"rolling_mean_{w}"] = gs.rolling(w).mean().reset_index(level=0, drop=True).astype("float32")
        df[f"rolling_stdv_{w}"] = gs.rolling(w).std().reset_index(level=0, drop=True).astype("float32")
    # new: same-weekday averages
    df["dow_mean_4"] = df[[f"sales_lag_{l}" for l in [7, 14, 21, 28]]].mean(axis=1).astype("float32")
    df["dow_mean_8"] = df[[f"sales_lag_{l}" for l in [7, 14, 21, 28, 35, 42, 49, 56]]].mean(axis=1).astype("float32")
    # new: short-term level (uses data up to yesterday, like sales_lag_1)
    df["_s1"] = g.shift(1)
    g1 = df.groupby("item_id", sort=False)["_s1"]
    for w in [3, 7, 28]:
        df[f"recent_mean_{w}"] = g1.rolling(w).mean().reset_index(level=0, drop=True).astype("float32")
    # new: intermittency descriptors (up to yesterday)
    df["_z"] = (df["_s1"] == 0).astype("float32").where(df["_s1"].notna())
    df["zero_share_28"] = df.groupby("item_id", sort=False)["_z"].rolling(28).mean().reset_index(level=0, drop=True).astype("float32")
    df["_nz"] = (df["_s1"] > 0).astype("float32").where(df["_s1"].notna())
    cnt = df.groupby("item_id", sort=False)["_nz"].rolling(28).sum().reset_index(level=0, drop=True)
    tot = g1.rolling(28).sum().reset_index(level=0, drop=True)
    df["nonzero_count_28"] = cnt.astype("float32")
    df["nonzero_mean_28"] = (tot / cnt.replace(0, np.nan)).fillna(0).astype("float32")
    last_sale = df["date"].where(df["sales"] > 0).groupby(df["item_id"], sort=False).ffill()
    last_sale_prev = last_sale.groupby(df["item_id"], sort=False).shift(1)
    df["days_since_sale"] = (df["date"] - last_sale_prev).dt.days.fillna(999).clip(upper=999).astype("float32")
    # new: item-level history encoding (expanding mean/std up to yesterday)
    csum = df["_s1"].fillna(0).groupby(df["item_id"], sort=False).cumsum()
    csq = (df["_s1"].fillna(0) ** 2).groupby(df["item_id"], sort=False).cumsum()
    n = df["_s1"].notna().groupby(df["item_id"], sort=False).cumsum().replace(0, np.nan)
    df["item_mean_hist"] = (csum / n).astype("float32")
    df["item_std_hist"] = np.sqrt((csq / n - (csum / n) ** 2).clip(lower=0)).astype("float32")
    df = df.drop(columns=["_sh", "_s1", "_z", "_nz"])
    # calendar / events
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["end_of_week"] = (df["day_of_week"] >= 5).astype(int)
    df["is_event"] = df["event_name_1"].notna().astype(int)
    # price (baseline) + price dynamics (new)
    gp = df.groupby("item_id", sort=False)["sell_price"]
    csp = df["sell_price"].fillna(0).groupby(df["item_id"], sort=False).cumsum()
    cnp = df["sell_price"].notna().groupby(df["item_id"], sort=False).cumsum()
    df["price_ratio_to_mean"] = (df["sell_price"] / (csp / cnp.replace(0, np.nan))).astype("float32")
    df["price_ratio_to_max"] = (df["sell_price"] / gp.cummax()).astype("float32")
    df["price_change_1w"] = (df["sell_price"] / gp.shift(7) - 1).astype("float32")
    df["price_rel_dept"] = (df["sell_price"] / df.groupby(["date", "dept_id"])["sell_price"].transform("mean")).astype("float32")
    return df

def feat_paths(s):
    return f"{FEATS_DIR}/{s}_train.parquet", f"{FEATS_DIR}/{s}_test.parquet"

for store_id in store_list:
    trp, tep = feat_paths(store_id)
    if os.path.exists(trp) and os.path.exists(tep):
        print(f"{store_id}: features ready"); continue
    raw = pd.read_parquet(f"{MERGED_DIR}/{store_id}.parquet")
    raw = raw[raw["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS + 200)]
    f = build_features_ext(raw); del raw; gc.collect()
    f[(f["date"] < test_start) & (f["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS))].to_parquet(trp, index=False)
    f[f["date"] >= test_start].to_parquet(tep, index=False)
    print(f"{store_id}: features built ({f.shape[1]} columns)"); del f; gc.collect()


## Step 3 - Shared functions

In [ ]:
def to_category(df):
    df = df.copy()
    for c in CAT_FEATURES:
        df[c] = df[c].astype("category")
    return df

def rmsse(y, p, hist):
    d = np.mean(np.diff(hist) ** 2) if len(hist) > 1 else 0
    return np.sqrt(np.mean((y - p) ** 2) / d) if d > 0 else np.nan

def sme(y, p, hist):
    d = np.mean(np.abs(np.diff(hist))) if len(hist) > 1 else 0
    return np.mean(y - p) / d if d > 0 else np.nan

def features_for(groups):
    return [c for gname in groups for c in FEATURE_GROUPS[gname]]

# rows are filtered exactly like the baseline (rows missing any base-paper input are not used),
# so every method is trained and scored on the same days of the same products
BASE_ROWS = features_for(BASELINE_GROUPS)

FIXED = {"subsample": 0.5, "subsample_freq": 1, "verbosity": -1, "seed": RANDOM_SEED, "n_jobs": -1}
DEFAULT_PARAMS = {"objective": "tweedie", "tweedie_variance_power": 1.3, "n_estimators": 250, "num_leaves": 63,
                  "min_data_in_leaf": 400, "learning_rate": 0.04, "feature_fraction": 0.6, "max_bin": 127,
                  "lambda_l1": 0.5, "lambda_l2": 0.5}

def store_params(store_id):
    """Use the hyperparameters already tuned for this store in Optimization 1 (if available), else defaults."""
    path = f"{OUT_DIR}/tuning_log.csv"
    if os.path.exists(path):
        log = pd.read_csv(path)
        row = log[(log["method"] == "Opt1 CMA-ES + pruning") & (log["group"] == store_id)]
        if len(row):
            r = row.iloc[-1]
            p = {k: (int(r[k]) if k in ["n_estimators", "num_leaves", "min_data_in_leaf", "max_bin"] else float(r[k]))
                 for k in DEFAULT_PARAMS if k != "objective"}
            p["objective"] = "tweedie"
            return p, "tuned in Optimization 1"
    return dict(DEFAULT_PARAMS), "default"

def fit(params, feats, tr, seed=RANDOM_SEED, weight=None):
    m = lgb.LGBMRegressor(**params, **{**FIXED, "seed": seed})
    m.fit(tr[feats + CAT_FEATURES], tr["sales"], sample_weight=weight, categorical_feature=CAT_FEATURES)
    return m

def split_train_valid(df):
    cut = df["date"].max() - pd.Timedelta(days=VALID_DAYS - 1)
    return df[df["date"] < cut], df[df["date"] >= cut]
print("Ready")



## Step 4 - Train Opt 6 per store and score it on the test period

In [ ]:
OPT4_BEST_GROUPS = ["lags_1_7", "lags_8_14", "rolling_h7", "calendar", "promo_event", "weekly_lags", "same_weekday_avg",
                    "intermittency", "short_rolling", "price_dynamics", "event_proximity", "item_encoding"]
search_file = f"{OUT_DIR}/optimization4_feature_search.csv"
if os.path.exists(search_file):
    t = pd.read_csv(search_file).sort_values("value").iloc[0]
    groups_from_file = [g for g in FEATURE_GROUPS if str(t.get(f"params_{g}", False)) in ("True", "1", "1.0")]
    if groups_from_file:
        OPT4_BEST_GROUPS = groups_from_file
best_feats = features_for(OPT4_BEST_GROUPS)
print("Opt 4 best feature groups:", OPT4_BEST_GROUPS, f"({len(best_feats)} inputs)")

results, preds = [], []
for s in store_list:
    t0 = time.time()
    tr_raw, te_raw = (pd.read_parquet(p) for p in feat_paths(s))
    # RMSSE scale: the same 365-day history (all rows) that every other method uses
    h = tr_raw[tr_raw["date"] >= test_start - pd.Timedelta(days=SCORE_HIST_DAYS)].sort_values("date")
    HIST = {k: g["sales"].values for k, g in h.groupby(KEY)}; del h
    # sales-value weight from the 28 days BEFORE the test period (no test information)
    last = tr_raw[tr_raw["date"] >= test_start - pd.Timedelta(days=28)]
    rev = (last["sales"] * last["sell_price"].fillna(0)).groupby(last["item_id"]).sum()
    floor = rev[rev > 0].min() if (rev > 0).any() else 1.0
    rev = rev.clip(lower=floor)
    wmap = np.sqrt(rev / rev.median())
    tr = to_category(tr_raw.dropna(subset=BASE_ROWS + ["sales"])); te = to_category(te_raw.dropna(subset=BASE_ROWS)); del tr_raw, te_raw; gc.collect()
    w = tr["item_id"].map(wmap).fillna(1.0).values
    params, src = store_params(s)
    p = np.zeros(len(te))
    for seed in SEEDS:
        m = fit(params, best_feats, tr, seed=seed, weight=w)
        p += m.predict(te[best_feats + CAT_FEATURES]) / len(SEEDS)
        del m; gc.collect()
    te = te.assign(pred=p)
    preds.append(te[KEY + ["date", "sales", "pred"]].assign(method="Opt 6: 2-year history + sales-value weights"))
    for k, gg in te.groupby(KEY):
        hh = HIST.get(k)
        if hh is None:
            continue
        results.append({"store_id": k[0], "item_id": k[1],
                        "rmsse": rmsse(gg["sales"].values, gg["pred"].values, hh),
                        "sme": sme(gg["sales"].values, gg["pred"].values, hh)})
    print(f"{s}: {len(tr):,} training rows, params {src}, done in {(time.time() - t0) / 60:.1f} min")
    del tr, te, HIST; gc.collect()

opt6_df = pd.DataFrame(results)
opt6_df.to_csv(f"{OUT_DIR}/optimization6_results.csv", index=False)
pd.concat(preds, ignore_index=True).to_csv(f"{OUT_DIR}/optimization6_predictions.csv.gz", index=False)
scored = opt6_df.dropna(subset=["rmsse"])
print(f"OPT 6 - {len(scored):,} products | median RMSSE {scored['rmsse'].median():.4f} | mean RMSSE {scored['rmsse'].mean():.4f}")
for label, path in [("Baseline", "baseline_dtf_results.csv"), ("Opt 4", "optimization4_feature_results.csv"), ("Opt 5", "optimization5_results.csv")]:
    pth = f"{OUT_DIR}/{path}"
    if os.path.exists(pth):
        cmp = pd.read_csv(pth).merge(scored[KEY + ["rmsse"]], on=KEY, suffixes=("_ref", "_opt6"))
        cmp = cmp[cmp["rmsse_ref"] < 1000]
        print(f"vs {label:8s} on {len(cmp):,} products: {label} mean {cmp['rmsse_ref'].mean():.4f} -> Opt 6 mean {cmp['rmsse_opt6'].mean():.4f} "
              f"| better on {(cmp['rmsse_opt6'] < cmp['rmsse_ref']).mean() * 100:.1f}% of products")


## Step 5 - Download the results

In [ ]:
files_out = ["optimization6_results.csv", "optimization6_predictions.csv.gz"]
zdir = f"{OUT_DIR}/_opt6zip"; os.makedirs(zdir, exist_ok=True)
for f in files_out:
    shutil.copy(f"{OUT_DIR}/{f}", zdir)
shutil.make_archive("/content/opt6_results", "zip", zdir)
from google.colab import files
files.download("/content/opt6_results.zip")
